# Mô hình cuối · CNN V2 + BatchNorm

## Vì sao chọn CNN V2 + BatchNorm?

Tiêu chí chính là **macro F1 trên tập validation**. Chỉ số này lấy trung bình F1 của 7 lớp cảm xúc với trọng số bằng nhau, nên những lớp có ít ảnh vẫn được tính ngang với các lớp phổ biến.

| Mô hình trong EXP03 | Accuracy | Macro F1 |
|---|---:|---:|
| Baseline + BatchNorm | 57,68% | 53,57% |
| Deeper CNN + BatchNorm | 59,94% | 54,91% |
| **CNN V2 + BatchNorm** | 60,54% | **56,05%** |
| CNN V3 + BatchNorm | **61,44%** | 55,53% |

V2 + BatchNorm có macro F1 cao nhất trong EXP03. V3 + BatchNorm dẫn đầu về accuracy, tức tỷ lệ ảnh được dự đoán đúng trên toàn tập. Theo tiêu chí ưu tiên cân bằng giữa các lớp, nhóm chọn V2 + BatchNorm.

Checkpoint `best.pth` của V2 được lưu ở epoch 4, khi validation loss của lần huấn luyện đó thấp nhất. Sau khi chốt mô hình bằng kết quả validation, chạy các ô bên dưới để dự đoán cảm xúc cho từng ảnh trong `data/test.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

cwd = Path.cwd()
root = next(p for p in (cwd, cwd.parent, cwd / "BTL02") if (p / "fer_data.py").exists())
sys.path.insert(0, str(root))
from fer_data import EMOTIONS, EmotionDataset, load_images, test_loader
from models import build_model

run = root / "results/EXP03/improved_cnn_v2_bn/20261125_092340_644784"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model("improved_cnn_v2_bn").to(device)
model.load_state_dict(torch.load(run / "best.pth", map_location=device, weights_only=True))
model.eval()
print(f"Checkpoint: {run.name}/best.pth | Thiết bị: {device}")

Checkpoint: 20261125_092340_644784/best.pth | Thiết bị: cuda


## Đọc ảnh test

Giữ thứ tự trong CSV và chuẩn hóa pixel như lúc huấn luyện. Khi dùng nhãn từ FER2013 gốc, hàm `test_loader` kiểm tra từng ảnh khớp với dữ liệu tham chiếu.

In [2]:
data = root / "data"
test_csv = data / "test.csv"
has_labels = "emotion" in pd.read_csv(test_csv, nrows=0).columns or (data / "fer2013.csv").is_file()
if has_labels:
    loader = test_loader(data, batch_size=64)
else:
    images = load_images(pd.read_csv(test_csv, usecols=["pixels"])).astype("float32") / 255.0
    loader = DataLoader(EmotionDataset(images), batch_size=64, shuffle=False)
print(f"Số ảnh test: {len(loader.dataset)}")

Số ảnh test: 7178


## Dự đoán và đánh giá

Mỗi dòng kết quả gồm vị trí ảnh tính từ 0, mã lớp, tên cảm xúc và xác suất softmax của lớp dự đoán.

In [3]:
predicted, confidences, truth = [], [], []
with torch.inference_mode():
    for batch in loader:
        if has_labels:
            images, labels = batch
            truth.extend(labels.tolist())
        else:
            images = batch
        scores, classes = model(images.to(device)).softmax(dim=1).max(dim=1)
        predicted.extend(classes.cpu().tolist())
        confidences.extend(scores.cpu().tolist())

results = pd.DataFrame({
    "test_row": range(len(loader.dataset)),
    "predicted_id": predicted,
    "predicted_emotion": [EMOTIONS[i] for i in predicted],
    "confidence": confidences,
})
if has_labels:
    results["true_id"] = truth
    accuracy = accuracy_score(truth, predicted)
    macro_f1 = f1_score(truth, predicted, labels=range(len(EMOTIONS)), average="macro", zero_division=0)
    print(f"Test accuracy: {accuracy:.2%} | Test macro F1: {macro_f1:.2%}")

output = run / "test_predictions.csv"
results.to_csv(output, index=False)
print(f"Đã lưu {len(results)} dự đoán vào {output.relative_to(root)}")

Test accuracy: 60.11% | Test macro F1: 56.36%
Đã lưu 7178 dự đoán vào results\EXP03\improved_cnn_v2_bn\20261125_092340_644784\test_predictions.csv
